In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ.get("GEMINI_API_KEY"):
    print("Api Key Exists")
    print(os.getenv("GEMINI_API_KEY"))
else:
    print("Api Key not exists")

load_dotenv(override=True)

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [ ]:
# TASK -1 [Prompt]

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie summarizer"),
    ("human", "Please summarize the movie in brief : {input}")])

In [ ]:
# TASK - 2 [LLM]

llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.5-flash",temperature=0)

In [ ]:
# TASK - 3 [Str Parser]

str_parser = StrOutputParser()

In [ ]:
# TASK - 4 [Custom Runnable]
from langchain_core.runnables import RunnableLambda

def dictionary_maker(text:str)-> dict:

    return {"text" : text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker)

In [ ]:
# PARALLEL CHAIN 1

# TASK - 1 [Prompt]
linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for LinkedIn: {text}")])

# TASK - 2 [LLM]
llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.5-flash",temperature=0)

# TASK - 3 [Str Parser]
str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm_gemini | str_parser

In [ ]:
# PARALLEL CHAIN 2

from langchain_core.runnables import RunnableParallel, RunnableLambda

def insta_chain(text:dict):
    
    text = text["text"]

    # TASK - 1 [Prompt]
    insta_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for Instagram: {text}")])
    
    # TASK - 2 [LLM]
    llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.5-flash",temperature=0)

    # TASK - 3 [Str Parser]
    str_parser = StrOutputParser()

    chain_insta = insta_prompt | llm_gemini | str_parser

    result = chain_insta.invoke(text)

    return result

insta_chain_runnable = RunnableLambda(insta_chain)



In [ ]:
# Final Orchestration

final_chain = ( 
                    prompt_template | 
                    llm_openai | 
                    str_parser | 
                    dictionary_maker_runnable |
                    RunnableParallel(branches = {"linkedin": chain_linkedin, "instagram": insta_chain_runnable})
)


In [ ]:
final_chain.invoke("KGF")